## 01. Chapter 01 전처리 데이터 불러오기

### 📌 학습 목표 및 개념
* **개념:** Chapter 01에서 전처리를 완료하고 저장한 `book_bestseller_clean.csv` 데이터를 pandas로 불러와 데이터 형태와 결측치를 점검합니다.
* **학습 목적:**
  - 텍스트 벡터화(Vectorization) 작업 전, 분석에 사용할 '상품명' 컬럼이 정상적으로 로드되었는지 확인합니다.
  - 한글 데이터 깨짐 방지를 위해 `utf-8-sig` 인코딩 방식을 적용하고 데이터의 크기(Shape)를 검증합니다.

In [1]:

import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)

print("컬럼:", df_books.columns.tolist())

print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류
5,초역 부처의 말
6,청춘의 독서(특별증보판)
7,어른의 행복은 조용하다
8,채식주의자
9,단 한 번의 삶(강물에디션 활판인쇄 한정판)


### 💡 실습 결과 정리

* **결과 검증:**
  - `df_books.shape`를 통해 전체 도서 데이터 건수 및 컬럼 수 확인 완료
  - '상품명' 컬럼의 결측치(NA) 유무를 체크하여 텍스트 분석 준비 상태 점검
  - 한글 도서 제목이 깨지지 않고 정상적으로 로드됨을 확인
* **운용 팁:** CSV 파일 생성 시 사용한 인코딩 방식(`utf-8-sig`)과 동일하게 `read_csv`를 호출해야 한글 깨짐 현상을 방지할 수 있습니다.

## 02. 분석할 제목 문자열 준비하기

### 📌 학습 목표 및 개념
* **개념:** Vectorizer에 전달하기 위해 pandas Series 형태의 도서 제목 데이터를 문자열로 변환하고, 결측치(NaN) 및 공백 문자를 정제합니다.
* **학습 목적:**
  - `fillna("")`와 `astype(str)`을 사용하여 데이터 타입 불일치로 인한 오차 발생을 방지합니다.
  - 형태소 분석기를 적용하기 전, 원본 도서 제목 문자열 그대로 Vectorizer에 적용했을 때의 기본 동작 원리를 파악합니다.

In [2]:
# 1. 도서 제목 정제 및 문자열 처리
titles = (
    df_books["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 2. 빈 문자열 제거 및 인덱스 재설정
titles = titles[titles != ""].reset_index(drop=True)

# 3. 사용할 전체 제목 수 및 상위 10개 확인
print("사용할 제목 수:", len(titles))
titles.head(10)

사용할 제목 수: 199


0                      소년이 온다
1                          모순
2                  결국 국민이 합니다
3                         혼모노
4                          급류
5                    초역 부처의 말
6               청춘의 독서(특별증보판)
7                어른의 행복은 조용하다
8                       채식주의자
9    단 한 번의 삶(강물에디션 활판인쇄 한정판)
Name: 상품명, dtype: str

### 💡 실습 결과 정리

* **결과 검증:**
  - 결측치 및 빈 문자열이 완전히 제거된 정제 텍스트 데이터셋 생성 완료
  - `titles` 데이터의 총 건수 및 상위 샘플 10개를 확인하여 Vectorizer 입력 준비 완료
* **참고 사항:** 이번 단계에서는 Vectorizer의 기초 동작 방식을 이해하기 위해 원본 제목 문자열 그대로를 사용합니다.

## 03. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

### 📌 학습 목표 및 개념
* **개념:** 머신러닝 알고리즘은 텍스트 문자열을 직접 계산할 수 없으므로, 문장을 단어 단위로 분할하고 각 단어의 등장 빈도를 숫자로 변환하는 **텍스트 벡터화(Vectorization)** 개념을 이해합니다.
* **학습 목적:**
  - "데이터 분석을 위한 파이썬"과 같은 텍스트가 어떤 구조를 거쳐 숫자 배열(Vector) 형태로 바뀌는지 변환 흐름을 파악합니다.

In [3]:
# 텍스트 벡터화 개념 흐름 시각적 확인
vectorization_flow = [
    "1. 도서 제목 텍스트",
    "2. 사용된 단어 확인 (토큰화)",
    "3. 단어마다 열(Column) 생성",
    "4. 등장 여부 또는 등장 횟수를 숫자로 기록",
    "5. 숫자 벡터(Vector) 완성"
]

print("=== 텍스트 벡터화(Vectorization) 변환 흐름 ===")
for step in vectorization_flow:
    print(step)

=== 텍스트 벡터화(Vectorization) 변환 흐름 ===
1. 도서 제목 텍스트
2. 사용된 단어 확인 (토큰화)
3. 단어마다 열(Column) 생성
4. 등장 여부 또는 등장 횟수를 숫자로 기록
5. 숫자 벡터(Vector) 완성


### 💡 실습 결과 정리

* **핵심 원리:**
  - 문자열 데이터는 덧셈, 곱셈 등의 수학적 연산이 불가능함
  - 단어 사전(Vocabulary)을 구축하고 각 문장에서 해당 단어가 몇 번 등장했는지 숫자로 표현함으로써 머신러닝 모델의 입력값으로 활용 가능함
* **다음 단계:** 세 문장의 아주 작은 예제 데이터셋을 직접 만들어 Bag of Words(BoW)의 변환 과정을 확인해 봅니다.

## 04. 아주 작은 예제로 먼저 이해하기

### 📌 학습 목표 및 개념
* **개념:** 전체 데이터셋에 적용하기 전, 3개의 문장으로 구성된 소규모 데이터셋(`sample_docs`)을 통해 텍스트가 숫자로 변환되는 과정을 직관적으로 파악합니다.
* **학습 목적:**
  - 사람이 직접 단어를 추출하고 열(Column)을 배열해보는 과정을 거쳐, 이후 사용할 `CountVectorizer`의 내부 동작 방식을 이해합니다.

In [4]:
# 1. 테스트용 소규모 샘플 문장 정의
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

# 2. 사람이 직접 추출한 단어 목록 (단어 사전 역할)
vocab = ["데이터", "머신러닝", "분석", "입문", "파이썬"]

print("=== 샘플 문장 목록 ===")
for i, doc in enumerate(sample_docs, 1):
    print(f"문장 {i}: {doc}")

print("\n=== 단어 사전 목록 (열 순서) ===")
print(vocab)

=== 샘플 문장 목록 ===
문장 1: 파이썬 데이터 분석
문장 2: 파이썬 머신러닝
문장 3: 데이터 분석 입문

=== 단어 사전 목록 (열 순서) ===
['데이터', '머신러닝', '분석', '입문', '파이썬']


In [8]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. 단어 사전 및 벡터 변환
count_vectorizer = CountVectorizer()
X_count_sample = count_vectorizer.fit_transform(sample_docs)

# 2. 열(Column)로 사용할 단어 이름 가져오기
feature_names = count_vectorizer.get_feature_names_out()

# 3. 튜터님 화면처럼 표(DataFrame)로 변환해서 출력
sample_count_df = pd.DataFrame(
    X_count_sample.toarray(), # 희소 행렬을 일반 배열로 변환
    columns=feature_names,
    index=sample_docs
)

sample_count_df

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


### 💡 실습 결과 정리

* **결과 검증:**
  - 3개의 문장에서 중복을 제외한 총 5개의 단어(`데이터`, `머신러닝`, `분석`, `입문`, `파이썬`)를 추출하여 단어 사전의 열(Column)을 정의함
  - 첫 번째 문장 `"파이썬 데이터 분석"`을 위의 단어 사전 순서에 맞춰 숫자로 표현하면 `[1, 0, 1, 0, 1]` 형태의 5차원 숫자 벡터로 나타낼 수 있음

## 05. Bag of Words 이해하기

### 📌 학습 목표 및 개념
* **개념:** **Bag of Words(BoW)**는 단어들의 순서나 문맥 정보는 무시하고, 문서에 등장하는 단어들을 하나의 가방(Bag) 속에 담아 **단어가 몇 번 등장했는지(Count)**만 세어 텍스트를 숫자로 표현하는 방식입니다.
* **학습 목적:**
  - "파이썬 데이터 분석"과 "데이터 파이썬 분석"처럼 단어 순서가 달라도 동일한 Count 벡터로 표현된다는 한계점과 특징을 이해합니다.
  - 다음 실습에서 사용할 scikit-learn의 `CountVectorizer`가 기반하고 있는 핵심 알고리즘 원리를 파악합니다.

In [6]:
# Bag of Words(BoW) 특징을 확인하는 예시 문장
bow_example = {
    "문장 1": "파이썬 데이터 분석",
    "문장 2": "데이터 파이썬 분석"
}

print("=== Bag of Words(BoW) 단어 순서 무시 예시 ===")
for name, text in bow_example.items():
    print(f"{name}: '{text}'")

print("\n👉 두 문장은 단어 순서가 다르지만, BoW 방식에서는 같은 단어가 동일한 횟수(파이썬:1, 데이터:1, 분석:1)로 등장하므로 완전히 동일한 숫자 벡터가 됩니다.")

=== Bag of Words(BoW) 단어 순서 무시 예시 ===
문장 1: '파이썬 데이터 분석'
문장 2: '데이터 파이썬 분석'

👉 두 문장은 단어 순서가 다르지만, BoW 방식에서는 같은 단어가 동일한 횟수(파이썬:1, 데이터:1, 분석:1)로 등장하므로 완전히 동일한 숫자 벡터가 됩니다.


### 💡 실습 결과 정리

* **핵심 특징:**
  - 단어의 순서(Order)나 문맥(Context) 정보는 제외하고, 오직 **단어의 출현 빈도수**만 계산함
  - 구조가 매우 간단하여 텍스트 분류, 문서 유사도 측정 등 다양한 머신러닝 문제의 기본 Baseline으로 유용하게 활용됨
* **다음 단계:** scikit-learn 라이브러리의 `CountVectorizer`를 불러와서 앞서 작성한 `sample_docs` 예제에 직접 적용해 봅니다.

## 06~07. CountVectorizer 라이브러리 및 작은 예제 적용하기

### 📌 학습 목표 및 개념
* **개념:** `CountVectorizer`는 텍스트 문서 집합을 단어 빈도 카운트 행렬로 변환해 주는 scikit-learn의 핵심 클래스입니다.
* **학습 목적:**
  - `fit_transform()` 함수가 수행하는 두 단계 동작(`fit`: 단어 사전 구축, `transform`: 문장을 숫자 벡터로 변환)의 작동 원리를 이해합니다.

In [9]:
# 1. CountVectorizer 모듈 임포트
from sklearn.feature_extraction.text import CountVectorizer

# 2. CountVectorizer 객체 생성 및 샘플 데이터에 적용 (fit_transform)
count_vectorizer = CountVectorizer()
X_count_sample = count_vectorizer.fit_transform(sample_docs)

print("변환 결과 객체 타입:", type(X_count_sample))
print("행렬의 크기 (문서 수, 단어 수):", X_count_sample.shape)

변환 결과 객체 타입: <class 'scipy.sparse._csr.csr_matrix'>
행렬의 크기 (문서 수, 단어 수): (3, 5)


### 💡 실습 결과 정리

* **결과 검증:**
  - `X_count_sample`의 형태가 `(3, 5)`로 출력되며, 3개의 문장이 5개의 단어를 기준 열로 하는 숫자 행렬로 변환됨을 확인
  - 반환 타입이 메모리를 효율적으로 쓰는 **희소 행렬(Sparse Matrix)** 타입임을 확인
* **핵심 동작 방식:**
  - `fit()`: 3개 문장을 읽고 중복 없는 단어 사전(Vocabulary) 구축
  - `transform()`: 구축된 단어 사전을 기준으로 각 문장의 단어 등장 횟수 카운트

## 08~09. 생성된 단어 사전 및 단어-문서 행렬 확인하기

### 📌 학습 목표 및 개념
* **개념:** `get_feature_names_out()`으로 추출한 단어 목록을 열(Column)로 지정하고, 희소 행렬을 `toarray()`로 변환하여 pandas DataFrame 형태의 **단어-문서 행렬(Document-Term Matrix)**을 생성합니다.
* **학습 목적:**
  - 행(Row)은 문장, 열(Column)은 단어, 값(Value)은 단어 등장 횟수임을 확인합니다.
  - 숫자 벡터의 각 자리가 단어 사전의 어떤 단어와 매핑되는지 직접 확인하고 해석하는 감각을 익힙니다.

In [10]:
import pandas as pd

# 1. Vectorizer가 학습한 단어 사전(열 이름) 추출
feature_names = count_vectorizer.get_feature_names_out()
print("추출된 단어 사전:", feature_names)

# 2. 희소 행렬을 배열로 변환하여 DataFrame 생성
sample_count_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=feature_names,
    index=sample_docs
)

# 3. 단어-문서 행렬(표) 출력
sample_count_df

추출된 단어 사전: ['데이터' '머신러닝' '분석' '입문' '파이썬']


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


### 💡 실습 결과 정리

* **결과 검증:**
  - `['데이터', '머신러닝', '분석', '입문', '파이썬']` 순서로 단어 사전이 구축됨을 확인
  -첫 번째 문장 `"파이썬 데이터 분석"` 행에서 `데이터(1)`, `분석(1)`, `파이썬(1)`에 각각 1이 들어간 정수 행렬이 완성됨
* **핵심 구조:**
  - **행(Row):** 입력받은 문서(도서 제목)
  - **열(Column):** 단어 사전의 특징 단어
  - **값(Value):** 해당 문서에서 해당 단어가 등장한 횟수

## 10. 같은 단어가 여러 번 나오면 어떻게 될까?

### 📌 학습 목표 및 개념
* **개념:** `CountVectorizer`는 단어의 단순 존재 여부(0 또는 1)뿐만 아니라, **문장 내 단어의 등장 횟수(Frequency)**를 정확히 카운트하여 숫자로 기록합니다.
* **학습 목적:**
  - 동일한 단어가 한 문장에 여러 번 등장할 경우(예: `"파이썬 파이썬 데이터"`) 해당 위치의 값이 2 이상으로 누적되는지 확인합니다.

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. 단어가 중복 등장하는 테스트 예제 작성
repeat_docs = [
    "파이썬 파이썬 데이터",
    "데이터 분석",
]

# 2. CountVectorizer 학습 및 변환
repeat_vectorizer = CountVectorizer()
X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

# 3. DataFrame으로 변환하여 등장 횟수 확인
repeat_df = pd.DataFrame(
    X_repeat.toarray(),
    columns=repeat_vectorizer.get_feature_names_out(),
    index=repeat_docs
)
repeat_df

,데이터,분석,파이썬
파이썬 파이썬 데이터,1,0,2
데이터 분석,1,1,0


### 💡 실습 결과 정리

* **결과 검증:**
  - 첫 번째 문장 `"파이썬 파이썬 데이터"` 행에서 `파이썬` 컬럼의 값이 **2**로 기록됨을 확인
  - `CountVectorizer`가 단순 binary(0/1) 방식이 아니라 단어의 중복 출현 횟수를 정수로 카운트함을 입증
* **참고 사항:** 만약 등장 여부(0 또는 1)만 필요하다면 `CountVectorizer(binary=True)` 옵션을 지정하여 사용할 수 있습니다.

## 11. CountVectorizer의 기본 토큰 기준 확인하기

### 📌 학습 목표 및 개념
* **개념:** `CountVectorizer`는 기본적으로 두 글자 이상의 단어만 토큰으로 인식하며, 기본 설정에서는 한 글자 단어(예: 'R', '책' 등)가 분석 대상에서 제외될 수 있습니다.
* **학습 목적:**
  - `token_pattern` 기본 옵션에 의해 한 글자 토큰이 어떻게 처리되는지 직접 확인합니다.
  - 전처리 조건에 따라 생성되는 단어 사전(Vocabulary)이 달라짐을 이해합니다.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. 한 글자 단어가 포함된 테스트 문장 작성
test_docs = [
    "AI 데이터 분석 R 파이썬",
]

# 2. 기본 CountVectorizer 학습 및 변환
test_vectorizer = CountVectorizer()
X_test = test_vectorizer.fit_transform(test_docs)

# 3. 추출된 단어 목록 확인
print("추출된 단어 목록:", test_vectorizer.get_feature_names_out())

추출된 단어 목록: ['ai' '데이터' '분석' '파이썬']


### 💡 실습 결과 정리

* **결과 검증:**
  - 문장 속 'R'과 같은 한 글자 단어는 제외되고 `['ai', '데이터', '분석', '파이썬']`만 단어 사전으로 추출됨을 확인
  - 영문 대문자("AI")가 소문자("ai")로 자동 변환(lowercase=True 기본 적용)됨을 확인
* **운용 팁:** 한 글자 단어도 분석에 포함해야 하는 경우, `CountVectorizer(token_pattern=r"(?u)\b\w+\b")` 옵션을 지정하여 사용할 수 있습니다.

## 12. 실제 도서 제목에 CountVectorizer 적용하기

### 📌 학습 목표 및 개념
* **개념:** 소규모 예제에서 확인한 `fit_transform()`을 전체 도서 제목 데이터셋(`titles`)에 적용하여 대규모 **단어-문서 행렬(Document-Term Matrix)**을 생성합니다.
* **학습 목적:**
  - 전체 문장 수(행)와 추출된 중복 없는 전체 단어 수(열)로 이루어진 `X_count.shape`의 구조를 파악합니다.
  - 실제 데이터에 적용했을 때 차원(Dimension)이 어떻게 커지는지 확인합니다.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. 실제 도서 제목 데이터에 CountVectorizer 적용
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(titles)

# 2. 생성된 단어 사전 목록 가져오기
count_terms = count_vectorizer.get_feature_names_out()

# 3. 행렬 크기 및 구조 확인
print("문서 수 (전체 도서 제목 수):", X_count.shape[0])
print("단어 수 (추출된 단어 사전 크기):", X_count.shape[1])
print("전체 행렬 크기 (shape):", X_count.shape)

문서 수 (전체 도서 제목 수): 199
단어 수 (추출된 단어 사전 크기): 536
전체 행렬 크기 (shape): (199, 536)


### 💡 실습 결과 정리

* **결과 검증:**
  - `X_count.shape`의 첫 번째 값은 전체 도서 제목 건수, 두 번째 값은 추출된 단어의 총개수를 의미함
  - 실제 데이터셋 전체를 단어 등장 빈도 기준의 숫자 벡터 공간으로 변환 완료
* **구조적 의미:**
  - **`X_count.shape[0]`**: 분석 대상 도서 수
  - **`X_count.shape[1]`**: 분석 모델이 학습한 전체 특징(Feature) 단어 수

## 13. vocabulary_ 확인하기

### 📌 학습 목표 및 개념
* **개념:** `CountVectorizer.vocabulary_`는 단어를 키(Key)로, 해당 단어가 단어-문서 행렬에서 위치한 열 인덱스 번호를 값(Value)으로 가지는 사전(Dictionary) 구조입니다.
* **학습 목적:**
  - `vocabulary_`의 숫자는 단어의 '등장 횟수'가 아니라 **'행렬 내 열 위치(인덱스 번호)'**를 의미한다는 점을 명확히 구분합니다.

In [14]:
# 1. vocabulary_ 사전의 상위 20개 단어 및 열 번호 대응 관계 확인
vocab_sample = list(count_vectorizer.vocabulary_.items())[:20]
print("=== vocabulary_ 상위 20개 샘플 (단어: 열 인덱스) ===")
for term, index in vocab_sample:
    print(f"'{term}': {index}번 열")

=== vocabulary_ 상위 20개 샘플 (단어: 열 인덱스) ===
'소년이': 278번 열
'온다': 355번 열
'모순': 194번 열
'결국': 67번 열
'국민이': 84번 열
'합니다': 511번 열
'혼모노': 524번 열
'급류': 92번 열
'초역': 460번 열
'부처의': 236번 열
'청춘의': 458번 열
'독서': 150번 열
'특별증보판': 485번 열
'어른의': 332번 열
'행복은': 517번 열
'조용하다': 432번 열
'채식주의자': 454번 열
'번의': 220번 열
'강물에디션': 57번 열
'활판인쇄': 528번 열


### 💡 실습 결과 정리

* **결과 검증:**
  - 단어마다 부여된 정수 값은 단어-문서 행렬에서 해당 단어가 위치한 **열 번호(Column Index)**임을 확인
* **⚠️ 혼동 주의:**
  - **`vocabulary_`의 숫자:** 해당 단어의 열 인덱스 번호 (0, 1, 2, ...)
  - **행렬(Matrix) 내부의 숫자:** 해당 문서에서 그 단어가 실제로 등장한 횟수

## 14. 희소 행렬(Sparse Matrix) 이해하기

### 📌 학습 목표 및 개념
* **개념:** **희소 행렬(Sparse Matrix)**은 행렬 내 대부분의 원소가 0인 행렬을 의미하며, 0이 아닌 유효한 값과 위치 정보만 메모리에 저장하여 효율성을 높이는 데이터 구조입니다.
* **학습 목적:**
  - `X_count`의 데이터 타입을 확인하여 scipy의 희소 행렬 형태로 저장되었음을 확인합니다.
  - 대용량 데이터에서 전체 행렬을 일반 배열(`toarray()`)로 함부로 변환할 경우 메모리가 부족해질 수 있음을 이해합니다.

In [15]:
# 1. 반환된 Count 행렬의 객체 타입 확인
print("X_count의 데이터 타입:", type(X_count))

# 2. 전체 데이터 중 0이 아닌 실제 값의 개수(Non-zero count) 확인
print("0이 아닌 실제 값이 들어있는 개수:", X_count.nnz)

# 3. 전체 셀 중에서 0이 아닌 값의 비율 계산 (%)
total_cells = X_count.shape[0] * X_count.shape[1]
nonzero_ratio = (X_count.nnz / total_cells) * 100
print(f"전체 셀 중 유효 데이터 비율: {nonzero_ratio:.2f}%")

X_count의 데이터 타입: <class 'scipy.sparse._csr.csr_matrix'>
0이 아닌 실제 값이 들어있는 개수: 684
전체 셀 중 유효 데이터 비율: 0.64%


In [17]:
import pandas as pd

# 1. 희소 행렬 정보 집계
total_titles = X_count.shape[0]
total_words = X_count.shape[1]
total_cells = total_titles * total_words
nonzero_cells = X_count.nnz
sparsity = (1 - (nonzero_cells / total_cells)) * 100

# 2. 교수님처럼 요약 정보 표(DataFrame) 생성
sparse_info_df = pd.DataFrame({
    "구분": ["전체 도서 수(행)", "전체 단어 수(열)", "총 셀(Cell) 수", "유효 데이터 수", "0의 비율(%)"],
    "값": [f"{total_titles:,}개", f"{total_words:,}개", f"{total_cells:,}개", f"{nonzero_cells:,}개", f"{sparsity:.2f}%"]
})

sparse_info_df

,구분,값
0,전체 도서 수(행),199개
1,전체 단어 수(열),536개
2,총 셀(Cell) 수,"106,664개"
3,유효 데이터 수,684개
4,0의 비율(%),99.36%


### 💡 실습 결과 정리

* **결과 검증:**
  - `X_count`가 `scipy.sparse` 형태로 관리되어 메모리 낭비를 줄이고 있음을 확인
  - 전체 셀 중 99% 이상이 0으로 채워져 있으며, 실제 의미 있는 데이터는 1% 미만임을 숫자로 증명
* **⚠️ 실무 주의사항:**
  - 샘플 데이터와 달리 전체 데이터 `X_count`에 `X_count.toarray()`를 함부로 호출하면 메모리 초과(MemoryError)가 발생할 수 있으므로 주의해야 합니다.

## 15. 실제 데이터의 첫 번째 제목 벡터 확인하기

### 📌 학습 목표 및 개념
* **개념:** 전체 도서 데이터 중 첫 번째 도서 제목(`titles.iloc[0]`)이 `CountVectorizer`를 거쳐 어떤 단어와 빈도 수(숫자)로 변환되었는지 표(DataFrame) 형태로 추출하여 직접 검증합니다.
* **학습 목적:**
  - 희소 행렬의 첫 번째 행에서 0이 아닌 유효 데이터만 골라내어 단어명과 등장 횟수를 매핑합니다.
  - 원본 도서 제목과 추출된 결과 표를 대조하며 정확히 변환되었는지 직접 확인합니다.

In [18]:
import pandas as pd

# 1. 첫 번째 도서 제목 확인
first_title = titles.iloc[0]
print("👉 분석할 첫 번째 도서 제목:", first_title)

# 2. 첫 번째 도서의 Count 벡터 추출 (0이 아닌 단어만 선별)
first_row = X_count.getrow(0)
indices = first_row.indices
values = first_row.data

# 3. 교수님처럼 예쁜 표(DataFrame) 형태로 가공
first_title_df = pd.DataFrame({
    "단어": count_terms[indices],
    "등장횟수": values
}).sort_values(by="등장횟수", ascending=False).reset_index(drop=True)

# 4. 표 출력
first_title_df

👉 분석할 첫 번째 도서 제목: 소년이 온다


,단어,등장횟수
0,소년이,1
1,온다,1


### 💡 실습 결과 정리

* **결과 검증:**
  - 원본 제목에 포함된 단어들이 표의 '단어' 컬럼에 정확하게 매핑되고, 등장 횟수가 정수(1 이상)로 출력됨을 확인
  - 0인 단어들은 제외되어 첫 번째 도서 제목을 구성하는 핵심 단어만 표 형태로 깔끔하게 정리됨
* **직접 검증 체크리스트:**
  - 원본 도서 제목에 실제로存在する 단어인가?
  - 단어 수 카운트가 올바르게 수행되었는가?

## 16. 전체 데이터에서 많이 등장한 단어 확인하기

### 📌 학습 목표 및 개념
* **개념:** `X_count` 행렬의 각 열(Column)을 열 방향(`axis=0`)으로 모두 더하면 전체 도서 제목에서 각 단어가 등장한 **총 등장 횟수**를 계산할 수 있습니다.
* **학습 목적:**
  - `numpy` 및 `pandas`를 활용해 단어별 총 등장 횟수를 집계하고, 내림차순 정렬하여 **상위 빈도 단어 표(DataFrame)**를 생성합니다.

In [19]:
import numpy as np
import pandas as pd

# 1. 전체 문서에서 각 단어가 등장한 총 횟수 계산 (열 방향 합계)
count_sums = np.array(X_count.sum(axis=0)).ravel()

# 2. 교수님처럼 예쁜 단어 빈도 요약 표(DataFrame) 생성
count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums
})

# 3. 전체 등장 횟수 기준 내림차순 정렬 후 상위 30개 확인
count_summary_top30 = count_summary.sort_values(
    by="전체등장횟수", 
    ascending=False
).reset_index(drop=True).head(30)

# 4. 상위 30개 표 출력
count_summary_top30

,단어,전체등장횟수
0,에디션,12
1,리커버,8
2,기념,8
3,위한,7
4,해커스,7
5,토익,6
6,나는,5
7,내가,5
8,스페셜,5
9,싶은,4


### 💡 실습 결과 정리

* **결과 검증:**
  - 전체 베스트셀러 도서 제목에서 가장 자주 쓰인 핵심 단어 TOP 30 목록과 등장 횟수가 깔끔한 표(DataFrame) 형태로 정렬되어 출력됨
* **Chapter 02 결과와의 차이점 분석:**
  - Chapter 02(형태소 분석)와 등장 횟수가 완전히 일치하지 않을 수 있음
  - **이유:** Chapter 02는 Kiwi 형태소 분석기로 품사/불용어를 정제했지만, `CountVectorizer`는 띄어쓰기 기준의 기본 토큰화 패턴(2글자 이상)을 사용하기 때문입니다.

## 17. Count 상위 단어 저장하기

### 📌 학습 목표 및 개념
* **개념:** 앞서 집계한 단어 빈도 요약 표에서 상위 30개 단어를 추출하여 파일(`chapter03_count_top_terms.csv`)로 저장합니다.
* **학습 목적:**
  - `to_csv()` 함수를 사용해 전처리 및 분석 결과를 파일로 내보내는 방법을 익힙니다.
  - `encoding="utf-8-sig"` 옵션을 사용해 Excel이나 다른 프로그램에서 한글 깨짐 없이 열 수 있도록 처리합니다.

In [20]:
import pandas as pd

# 1. 상위 30개 단어 추출
count_top30 = count_summary_top30.head(30)

# 2. CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig 적용)
count_top30.to_csv("chapter03_count_top_terms.csv", index=False, encoding="utf-8-sig")

# 3. 저장된 CSV 파일 다시 불러와서 표(DataFrame)로 확인
saved_count_df = pd.read_csv("chapter03_count_top_terms.csv", encoding="utf-8-sig")
saved_count_df.head(10)

,단어,전체등장횟수
0,에디션,12
1,리커버,8
2,기념,8
3,위한,7
4,해커스,7
5,토익,6
6,나는,5
7,내가,5
8,스페셜,5
9,싶은,4


### 💡 실습 결과 정리

* **결과 검증:**
  - `chapter03_count_top_terms.csv` 파일이 정상적으로 생성 및 저장되었음을 확인
  - 저장된 파일을 다시 `read_csv`로 불러왔을 때 한글과 숫자 데이터가 깨짐 없이 표 형태로 잘 출력되는지 검증 완료

## 18. Count 방식의 한계 생각해 보기

### 📌 학습 목표 및 개념
* **개념:** 모든 문서에 흔하게 등장하는 단어(예: '책', '도서', '위한')는 등장 횟수(Count)가 높지만, **특정 문서의 주제나 특징을 구분하는 데는 기여도가 낮다**는 문제점을 이해합니다.
* **학습 목적:**
  - 단순히 단어가 많이 나왔다고 해서 중요도를 높게 평가하는 Count 방식의 한계를 파악합니다.
  - "흔한 단어의 가중치는 낮추고, 특정 문서에만 두드러지게 나오는 단어의 가중치는 높이는" **TF-IDF(Term Frequency - Inverse Document Frequency)** 개념의 필요성을 연결합니다.

In [21]:
import pandas as pd

# Count 방식의 한계를 보여주는 예시 데이터 생성
count_limitation_df = pd.DataFrame({
    "단어": ["도서", "파이썬", "분석"],
    "전체 문서 등장 횟수": [1000, 15, 20],
    "문서 구분 능력": ["매우 낮음 (모든 책 제목에 들어감)", "높음 (특정 기술 도서에만 등장)", "높음 (데이터 관련 도서에만 등장)"],
    "Count 방식 평가": ["가장 중요하다고 판단 (한계)", "상대적으로 낮게 평가됨", "상대적으로 낮게 평가됨"],
    "TF-IDF 방식 평가": ["가중치 감점 (IDF 낮음)", "가중치 부여 (IDF 높음)", "가중치 부여 (IDF 높음)"]
})

count_limitation_df

,단어,전체 문서 등장 횟수,문서 구분 능력,Count 방식 평가,TF-IDF 방식 평가
0,도서,1000,매우 낮음 (모든 책 제목에 들어감),가장 중요하다고 판단 (한계),가중치 감점 (IDF 낮음)
1,파이썬,15,높음 (특정 기술 도서에만 등장),상대적으로 낮게 평가됨,가중치 부여 (IDF 높음)
2,분석,20,높음 (데이터 관련 도서에만 등장),상대적으로 낮게 평가됨,가중치 부여 (IDF 높음)


### 💡 실습 결과 정리

* **핵심 한계점 정리:**
  - **Count 방식:** 단어의 출현 빈도만 계산하므로 모든 문서에 공통으로 들어가는 일반적인 단어가 상위를 독점함
  - **개선 방향 (TF-IDF):** 
    1. **TF (Term Frequency):** 현재 문서 안에서 단어가 얼마나 자주 등장하는가?
    2. **IDF (Inverse Document Frequency):** 전체 문서 집합 중에서 이 단어가 얼마나 희귀(드문)한가?
  - 이 두 가지를 곱해 **문서를 대표하는 진짜 핵심 단어**를 선별합니다.

## 19~22. TF, DF, IDF 및 TF-IDF 개념 한눈에 정리하기

### 📌 학습 목표 및 개념
* **개념:**
  - **TF (Term Frequency):** 현재 문서 안에서 특정 단어가 등장하는 빈도 (높을수록 해당 문서에서 중요)
  - **DF (Document Frequency):** 해당 단어가 등장하는 전체 문서의 개수
  - **IDF (Inverse DF):** DF의 역수 개념으로, 흔한 단어일수록 값이 낮아지고 희귀한 단어일수록 값이 커짐
  - **TF-IDF:** $TF \times IDF$ 형태로 두 값을 곱해 **"이 문서에서는 자주 나오지만 전체에서는 드문 핵심 단어"**에 높은 가중치를 부여

In [22]:
import pandas as pd

# TF-IDF 구성 요소를 한눈에 비교하는 요약 표 생성
tfidf_concept_df = pd.DataFrame({
    "용어": ["TF (Term Frequency)", "DF (Document Frequency)", "IDF (Inverse DF)", "TF-IDF"],
    "핵심 질문": [
        "이 문서 안에서 얼마나 자주 나왔는가?",
        "전체 문서 중 몇 개 문서에 등장하는가?",
        "전체 문서에서 얼마나 희귀(드문)한가?",
        "이 문서의 특징을 설명하는 진짜 핵심 단어인가?"
    ],
    "특징": [
        "문서 내 빈도가 높을수록 점수 상승",
        "단어가 널리 쓰일수록 값이 커짐",
        "DF가 높을수록(흔할수록) 감점, 낮을수록(희귀할수록) 가중치 부여",
        "TF와 IDF를 곱해 흔한 단어(공통어)의 영향력을 차단"
    ]
})

tfidf_concept_df

,용어,핵심 질문,특징
0,TF (Term Frequency),이 문서 안에서 얼마나 자주 나왔는가?,문서 내 빈도가 높을수록 점수 상승
1,DF (Document Frequency),전체 문서 중 몇 개 문서에 등장하는가?,단어가 널리 쓰일수록 값이 커짐
2,IDF (Inverse DF),전체 문서에서 얼마나 희귀(드문)한가?,"DF가 높을수록(흔할수록) 감점, 낮을수록(희귀할수록) 가중치 부여"
3,TF-IDF,이 문서의 특징을 설명하는 진짜 핵심 단어인가?,TF와 IDF를 곱해 흔한 단어(공통어)의 영향력을 차단


## 19~20. TF와 DF 개념 깊이 알아보기

### 1. TF (Term Frequency, 단어 빈도)
* **정의:** **하나의 특정 문서(문장)** 내에서 특정 단어가 얼마나 자주 등장하는지를 나타내는 값입니다.
* **핵심 질문:** *"이 문서 안에서 이 단어가 얼마나 강조되고 있는가?"*
* **특징 및 해석:**
  - 같은 문서 내에서 반복 사용된 단어일수록 TF 값이 높아집니다.
  - 예: `"파이썬 파이썬 데이터 분석"`이라는 도서 제목이 있다면, 이 문장 안에서는 `'파이썬'`의 TF 가중치가 가장 높게 평가됩니다.
  - **한계:** 특정 단어가 한 문서에서 아무리 많이 나와도, 그 단어가 모든 문서에 다 들어가는 흔한 단어라면 해당 문서만의 독자적인 특징이 되기 어렵습니다.

---

### 2. DF (Document Frequency, 문서 빈도)
* **정의:** 특정 단어가 **전체 데이터셋 중 몇 개의 문서(문장)**에 등장하는지 세어본 개수입니다.
* **핵심 질문:** *"이 단어가 전체 문서 전체에 얼마나 범용적으로 퍼져 있는가?"*
* **특징 및 해석:**
  - DF는 특정 문서 내 등장 횟수와 상관없이, **해당 단어를 포함하는 문서의 총개수**만 집계합니다.
  - **DF가 매우 높은 단어 예시:** `'책'`, `'도서'`, `'위한'`, `'가이드'`
    - 전체 1,000개 도서 제목 중 800개에 등장하므로 DF = 800입니다. (너무 흔해서 변별력 없음)
  - **DF가 낮은 단어 예시:** `'양자역학'`, `'파이토치'`, `'스프링부트'`
    - 전체 1,000개 도서 제목 중 3개에만 등장하므로 DF = 3입니다. (특정 전문 분야를 명확히 설명함)

---

### 📊 TF vs DF 한눈에 비교하기

| 구분 | TF (Term Frequency) | DF (Document Frequency) |
| :--- | :--- | :--- |
| **관점** | **단일 문서 내부** 관점 | **전체 문서 집합** 관점 |
| **측정 대상** | 한 문서 안에서의 단어 등장 횟수 | 단어가 등장하는 문서의 개수 |
| **의미** | 해당 문서에서의 국소적 중요도 | 해당 단어의 범용성 및 흔함 정도 |
| **TF-IDF에서의 역할** | 값이 클수록 **가중치 증가 (+)** | 값이 클수록 **가중치 감점 (-)** *(Inverse 적용)* |

### 💡 실습 결과 정리

* **핵심 한 줄 요약:**
  - **Count 방식:** 단순 등장 횟수만 세어 '책', '도서' 같은 흔한 단어가 상위를 독점함
  - **TF-IDF 방식:** 모든 문서에 흔하게 나오는 단어는 IDF로 감점시키고, 특정 주제 문서에만 집중 등장하는 단어에 높은 가중치를 부여함

## 23. 작은 예제로 TfidfVectorizer 적용하기

### 📌 학습 목표 및 개념
* **개념:** `TfidfVectorizer`는 `CountVectorizer`와 동일하게 단어 사전을 구축하고 문장을 벡터화하지만, 결과값을 단순 등장 횟수가 아닌 **TF-IDF 가중치(소수점 값)**로 계산합니다.
* **학습 목적:**
  - 동일한 샘플 데이터에 대해 Count 방식과 TF-IDF 방식의 결과 표를 함께 출력하여 수치적으로 어떤 변화가 일어나는지 직접 대조합니다.
  - 여러 문장에 흔하게 등장하는 단어와 특정 문장에만 등장하는 단어의 가중치 차이를 확인합니다.

In [23]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. 샘플 데이터 준비
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문"
]

# 2. CountVectorizer 적용 및 DataFrame 생성
count_vectorizer = CountVectorizer()
X_count_sample = count_vectorizer.fit_transform(sample_docs)
count_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=sample_docs
)

# 3. TfidfVectorizer 적용 및 DataFrame 생성
tfidf_vectorizer = TfidfVectorizer()
X_tfidf_sample = tfidf_vectorizer.fit_transform(sample_docs)
tfidf_df = pd.DataFrame(
    X_tfidf_sample.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=sample_docs
).round(3)

# 4. 결과 출력
print("=== 1. CountVectorizer 결과 (단순 빈도 정수) ===")
display(count_df)

print("\n=== 2. TfidfVectorizer 결과 (TF-IDF 가중치 소수) ===")
display(tfidf_df)

=== 1. CountVectorizer 결과 (단순 빈도 정수) ===


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0



=== 2. TfidfVectorizer 결과 (TF-IDF 가중치 소수) ===


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. 데이터 타입 및 표현 방식의 변화:**
  - **Count 방식:** 단어가 문장에 몇 번 등장했는지를 나타내는 **정수(1, 0)**로 출력됩니다.
  - **TF-IDF 방식:** 단어의 희귀도(IDF)와 문서 내 빈도(TF), 그리고 L2 정규화(벡터의 길이를 1로 맞춤)가 적용되어 **0~1 사이의 소수점 값**으로 변환됩니다.

* **2. 단어별 가중치 차이 상세 분석:**
  - **'입문' / '머신러닝' (특이 단어):** 3개 문장 중 단 1개의 문장에만 등장하므로 IDF 점수가 높습니다. 그 결과 해당 문장 내에서 **`0.671`**이라는 매우 높은 TF-IDF 가중치를 부여받아 문장의 핵심 키워드로 평가됩니다.
  - **'데이터' / '분석' / '파이썬' (흔한 단어):** 2개 문장에 걸쳐 비교적 자주 등장하기 때문에 상대적으로 IDF 점수가 깎여 **`0.518`** 수준의 가중치를 가지게 됩니다.

* **3. 결론 및 시사점:**
  - 단순 Count 방식에서는 모든 단어가 одина하게 1점으로 처리되어 중요한 단어를 가려내기 어렵지만, **TF-IDF를 적용하면 문장의 주제를 잘 나타내는 유일한 단어일수록 더 높은 점수를 획득**함을 수치로 증명할 수 있습니다.

## 24~25. TfidfVectorizer 단어 사전 및 단어별 IDF 값 확인하기

### 📌 학습 목표 및 개념
* **개념:** 
  - `get_feature_names_out()`: CountVectorizer와 동일하게 추출된 열(Column) 단어 목록을 반환합니다.
  - `idf_`: Vectorizer가 학습 과정에서 계산한 각 단어의 **IDF(역문서 빈도) 수치**를 보관하는 속성입니다.
* **학습 목적:**
  - 여러 문서에 두루 등장하는 단어와 특정 문서에만 드물게 등장하는 단어의 IDF 수치 차이를 **상세 표(DataFrame)**로 비교해 해석해 봅니다.

In [25]:
import pandas as pd

# 1. TfidfVectorizer가 만든 단어 목록 추출 (변수명 통일)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# 2. 단어별 IDF 값을 매핑하여 표(DataFrame) 생성
idf_df = pd.DataFrame({
    "단어": tfidf_terms,
    "IDF_점수": tfidf_vectorizer.idf_
}).sort_values(by="IDF_점수", ascending=False).reset_index(drop=True)

# 3. IDF 수치 내림차순 정렬 표 출력
idf_df

,단어,IDF_점수
0,머신러닝,1.693147
1,입문,1.693147
2,데이터,1.287682
3,분석,1.287682
4,파이썬,1.287682


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. IDF 수치와 단어 희귀도의 관계 분석:**
  - **'머신러닝' / '입문' (높은 IDF 점수 = 약 1.693):**
    - 3개의 샘플 문장 중 **단 1개 문장에만 등장**하는 희귀 단어입니다.
    - 전체 문장에서 드물게 출현하므로 **IDF 가중치가 높게 산출**되어 해당 문장을 대표하는 키워드로 강력하게 작동합니다.
  - **'데이터' / '분석' / '파이썬' (낮은 IDF 점수 = 약 1.288):**
    - 3개의 샘플 문장 중 **2개 이상의 문장에 걸쳐 흔하게 등장**하는 일반 단어입니다.
    - 범용적으로 자주 쓰이므로 **IDF 수치가 상대적으로 깎여(감점)** 가중치가 낮아집니다.

* **2. 핵심 시사점:**
  - IDF 점수는 *"이 단어가 전체 문서 집합 내에서 얼마나 희귀(드문)한가?"*를 나타내는 척도입니다.
  - TF-IDF는 이렇게 계산된 IDF 점수를 개별 문서의 TF(단어 빈도)와 곱해 최종 가중치를 부여하므로, **흔한 단어가 문장의 주제를 독점하는 현상을 효과적으로 방지**합니다.

## 26. 실제 도서 제목을 TF-IDF로 변환하기

### 📌 학습 목표 및 개념
* **개념:** `TfidfVectorizer`를 전체 도서 제목 데이터셋(`titles`)에 적용하여 대규모 **TF-IDF 단어-문서 행렬**을 생성합니다.
* **학습 목적:**
  - CountVectorizer로 만든 행렬(`X_count`)과 TfidfVectorizer로 만든 행렬(`X_tfidf`)의 크기(shape)를 직접 비교합니다.
  - 동일한 토큰화 기준을 사용했을 때 두 방식의 행렬 차원(행/열 수)이 일치함을 검증합니다.

In [27]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 실제 도서 제목 데이터에 TfidfVectorizer 적용 (변수명: tfidf_vectorizer)
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(titles)

# 2. 추출된 전체 단어 목록 가져오기
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# 3. Count 행렬과 TF-IDF 행렬 크기 비교 표(DataFrame) 생성
matrix_compare_df = pd.DataFrame({
    "행렬 종류": ["Count 행렬 (X_count)", "TF-IDF 행렬 (X_tfidf)"],
    "문서 수 (행)": [X_count.shape[0], X_tfidf.shape[0]],
    "단어 수 (열)": [X_count.shape[1], X_tfidf.shape[1]],
    "데이터 타입": [type(X_count).__name__, type(X_tfidf).__name__]
})

matrix_compare_df

,행렬 종류,문서 수 (행),단어 수 (열),데이터 타입
0,Count 행렬 (X_count),199,536,csr_matrix
1,TF-IDF 행렬 (X_tfidf),199,536,csr_matrix


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. Count 방식과 TF-IDF 방식의 구조적 비교:**
  - 두 방식 모두 동일한 기본 토큰화 조건(2글자 이상 단어)을 적용했기 때문에 **행(문서 수)과 열(단어 수)의 크기가 완전히 동일**하게 생성됩니다.
  - 차이점은 행렬 내부에 들어가는 **값(Value)**입니다. Count 방식은 '등장 횟수(정수)'가 들어가고, TF-IDF 방식은 '단어의 상대적 중요도 가중치(소수)'가 들어갑니다.

* **2. 메모리 관리 (희소 행렬):**
  - 두 방식 모두 `scipy.sparse` 형태의 **희소 행렬(Sparse Matrix)** 형태로 반환되어 메모리를 절약합니다.

## 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기

### 📌 학습 목표 및 개념
* **개념:** 전체 도서 제목 중 첫 번째 도서 제목(`titles.iloc[0]`)의 TF-IDF 벡터에서 0이 아닌 유효 단어와 해당 단어의 **TF-IDF 가중치(점수)**를 표 형태로 가공합니다.
* **학습 목적:**
  - 앞서 실습 15에서 확인한 Count 방식(단순 횟수 1, 1, ...)과 달리, TF-IDF 방식에서는 단어마다 가중치 점수가 다르게 부여됨을 직접 검증합니다.
  - TF-IDF 점수가 높은 순으로 정렬하여 가장 중요한 핵심 단어를 파악합니다.

In [28]:
import pandas as pd

# 1. 첫 번째 도서 제목 확인
first_title = titles.iloc[0]
print("👉 분석할 첫 번째 도서 제목:", first_title)

# 2. 첫 번째 도서의 TF-IDF 벡터 추출 (0이 아닌 단어만 선별)
first_tfidf_row = X_tfidf.getrow(0)
indices = first_tfidf_row.indices
values = first_tfidf_row.data

# 3. 예쁜 표(DataFrame) 형태로 가공 및 TF-IDF 점수 내림차순 정렬
first_tfidf_df = pd.DataFrame({
    "단어": tfidf_terms[indices],
    "TF-IDF_점수": values
}).sort_values(by="TF-IDF_점수", ascending=False).reset_index(drop=True)

# 4. 표 출력
first_tfidf_df

👉 분석할 첫 번째 도서 제목: 소년이 온다


,단어,TF-IDF_점수
0,소년이,0.707107
1,온다,0.707107


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. Count 방식과 TF-IDF 방식의 결과 비교:**
  - **Count 방식 (실습 15):** 모든 등장 단어가 동일하게 `1`점(등장 횟수)으로 표시되었습니다.
  - **TF-IDF 방식 (실습 27):** 단어마다 `0.3~0.7` 사이의 서로 다른 **가중치 소수점 값**이 부여됩니다.

* **2. 단어별 가중치 수치 해석:**
  - 전체 도서 데이터셋에서 상대적으로 **드물게 등장하는 희귀 단어일수록 더 높은 TF-IDF 점수**를 받습니다.
  - 여러 도서에 흔하게 겹치는 범용 단어는 가중치가 깎여 하위로 밀려나므로, **해당 도서만의 독자적인 주제를 나타내는 단어가 상위에 위치**하게 됩니다.

## 28. 전체 데이터의 TF-IDF 상위 단어 확인하기

### 📌 학습 목표 및 개념
* **개념:** 전체 도서 제목 데이터에서 각 단어가 갖는 TF-IDF 가중치의 총합을 계산하여, 단순 등장 횟수(Count)가 아닌 **가중치 합계 기준 상위 단어 목록 표(DataFrame)**를 생성합니다.
* **학습 목적:**
  - 앞서 실습 16에서 구한 Count 상위 단어 순위와 TF-IDF 상위 단어 순위를 대조해 봅니다.
  - 흔한 단어의 점수가 깎이고, 실제 도서 분야나 주제를 잘 드러내는 단어들의 순위가 어떻게 올라가는지 확인합니다.

In [29]:
import numpy as np
import pandas as pd

# 1. 전체 데이터셋에서 단어별 TF-IDF 가중치 총합 계산 (열 방향 합계)
tfidf_sums = np.array(X_tfidf.sum(axis=0)).ravel()

# 2. 예쁜 TF-IDF 요약 표(DataFrame) 생성
tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "TF-IDF_합계": tfidf_sums
})

# 3. TF-IDF 합계 기준 내림차순 정렬 후 상위 30개 추출
tfidf_summary_top30 = tfidf_summary.sort_values(
    by="TF-IDF_합계", 
    ascending=False
).reset_index(drop=True).head(30)

# 4. 표 출력
tfidf_summary_top30

,단어,TF-IDF_합계
0,에디션,3.846589
1,리커버,2.758350
2,기념,2.529548
3,위한,2.405380
4,해커스,2.402270
5,토익,2.179137
6,스페셜,1.970387
7,나는,1.947468
8,내가,1.881379
9,대하여,1.755504


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. Count 순위(실습 16) vs TF-IDF 순위(실습 28) 수치 비교:**
  - **Count 방식:** 단어가 많이 등장하기만 하면 상위권에 배치되어 '책', '도서' 같이 모든 곳에 흔하게 쓰이는 일반 단어가 1위를 차지하기 쉽습니다.
  - **TF-IDF 방식:** 여러 문서에 흔하게 퍼진 단어는 IDF 감점을 받아 가중치 합계 점수가 낮아지며, 특정 주제나 분야를 명확하게 드러내는 단어가 상대적으로 더 높게 평가됩니다.

* **2. 핵심 결론:**
  - TF-IDF를 사용하면 텍스트 데이터의 **"단순 빈도 노이즈"를 제거**하고, 데이터셋을 구성하는 **진짜 핵심 키워드**를 정확하게 추출해낼 수 있습니다.

## 29. TF-IDF 상위 단어 저장하기

### 📌 학습 목표 및 개념
* **개념:** 분석 결과로 나온 TF-IDF 상위 30개 단어 표를 CSV 파일(`chapter03_tfidf_top_terms.csv`)로 내보냅니다.
* **학습 목적:**
  - 앞서 실습 17에서 저장한 Count 상위 단어 파일과 이번 TF-IDF 상위 단어 파일을 비교·분석할 수 있도록 데이터를 파일 형태로 보관합니다.
  - `encoding="utf-8-sig"`를 지정해 Excel이나 다른 프로그램에서 한글이 깨지지 않도록 처리합니다.

In [30]:
import pandas as pd

# 1. TF-IDF 상위 30개 단어 추출
tfidf_top30 = tfidf_summary_top30.head(30)

# 2. CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig 적용)
tfidf_top30.to_csv("chapter03_tfidf_top_terms.csv", index=False, encoding="utf-8-sig")

# 3. 저장된 CSV 파일 불러와서 제대로 저장되었는지 표(DataFrame)로 출력
saved_tfidf_df = pd.read_csv("chapter03_tfidf_top_terms.csv", encoding="utf-8-sig")
saved_tfidf_df.head(10)

,단어,TF-IDF_합계
0,에디션,3.846589
1,리커버,2.758350
2,기념,2.529548
3,위한,2.405380
4,해커스,2.402270
5,토익,2.179137
6,스페셜,1.970387
7,나는,1.947468
8,내가,1.881379
9,대하여,1.755504


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. 파일 저장 및 데이터 검증:**
  - `chapter03_tfidf_top_terms.csv` 파일이 정상적으로 생성되고 저장되었음을 확인했습니다.
  - 다시 불러온 표 데이터에서 단어와 `TF-IDF_합계` 수치가 소수점 형태 그대로 깨짐 없이 깔끔하게 저장되었는지 검증했습니다.

* **2. Count 저장 결과(실습 17)와의 차이점:**
  - Count 파일에는 단어의 단순 **등장 횟수(정수)**가 저장되었으나, TF-IDF 파일에는 단어의 **상대적 중요도 가중치 합계(소수)**가 저장되어 텍스트 특징의 질적 차이를 한눈에 보여줍니다.

## 30. Count와 TF-IDF 결과 비교하기

### 📌 학습 목표 및 개념
* **개념:** 두 가지 벡터화 방식(Count vs TF-IDF)으로 추출한 상위 단어 목록을 하나의 통합 표(DataFrame)로 결합하여 비교합니다.
* **학습 목적:**
  - 단순 빈도(Count) 기준 상위 단어와 가중치(TF-IDF) 기준 상위 단어의 순위 차이를 한눈에 대조합니다.
  - TF-IDF가 흔하게 쓰이는 일반 단어의 순위를 어떻게 조정하고 핵심 키워드를 상위로 올리는지 수치적으로 검증합니다.

In [31]:
import pandas as pd

# 1. 저장해 둔 두 개의 CSV 파일 불러오기
df_count_top = pd.read_csv("chapter03_count_top_terms.csv", encoding="utf-8-sig")
df_tfidf_top = pd.read_csv("chapter03_tfidf_top_terms.csv", encoding="utf-8-sig")

# 2. 두 결과 표를 나란히 비교하기 위한 통합 DataFrame 생성 (상위 15개)
comparison_df = pd.DataFrame({
    "순위": range(1, 16),
    "Count 단어": df_count_top["단어"].head(15),
    "Count 등장횟수": df_count_top["전체등장횟수"].head(15),
    "TF-IDF 단어": df_tfidf_top["단어"].head(15),
    "TF-IDF 가중치합": df_tfidf_top["TF-IDF_합계"].head(15).round(3)
})

# 3. 비교 표 출력
comparison_df

,순위,Count 단어,Count 등장횟수,TF-IDF 단어,TF-IDF 가중치합
0,1,에디션,12,에디션,3.847
1,2,리커버,8,리커버,2.758
2,3,기념,8,기념,2.530
3,4,위한,7,위한,2.405
4,5,해커스,7,해커스,2.402
5,6,토익,6,토익,2.179
6,7,나는,5,스페셜,1.970
7,8,내가,5,나는,1.947
8,9,스페셜,5,내가,1.881
9,10,싶은,4,대하여,1.756


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. 두 방식의 상위 키워드 차이 분석:**
  - **Count 방식:** 문서 수와 상관없이 '단순히 많이 언급된 단어'가 무조건 상위권을 독점합니다.
  - **TF-IDF 방식:** 여러 도서 제목에 광범위하게 퍼져 있는 일반 단어는 IDF 감점을 받아 순위가 낮아지며, 특정 주제 분야를 뚜렷하게 설명하는 단어의 순위가 상승합니다.

* **2. 텍스트 전처리 및 표현 방식의 결론:**
  - 머신러닝 모델에 입력값을 넣을 때 단순 Count 형태보다 **TF-IDF 형태를 사용하는 것이 문서의 특성을 더 잘 반영하여 분류 및 분석 성능을 높이는 데 유리**합니다.

## 31~32. 데이터 누수(Data Leakage) 예방 원칙 및 Chapter 03 요약

### 📌 학습 목표 및 개념
* **데이터 누수 (Data Leakage):**
  - 모델 학습 전 전체 데이터(Train + Test)에 대해 미리 `fit()`을 수행하면, 테스트 데이터의 단어 집합이나 IDF 정보가 학습 과정에 스며드는 현상이 발생합니다.
  - **올바른 방식:** Train 데이터셋으로만 `fit_transform()`을 수행하고, Test 데이터셋에는 Train 데이터로 학습된 Vectorizer를 사용해 `transform()`만 적용해야 합니다.
* **Chapter 03 핵심 요약:**
  - **CountVectorizer:** 단어 등장 횟수(정수) 기반 벡터화 방식
  - **TfidfVectorizer:** $TF \times IDF$ 가중치(소수) 기반 벡터화 방식 (흔한 단어 감점, 특이 단어 우대)

In [32]:
import pandas as pd

# 1. 데이터 누수 방지 원칙 요약 표 생성
leakage_rule_df = pd.DataFrame({
    "구분": ["학습 데이터 (Train Data)", "테스트 데이터 (Test Data)"],
    "적용 메서드": ["fit_transform()", "transform()"],
    "이유 및 원칙": [
        "단어 사전을 구축(fit)함과 동시에 숫자 벡터로 변환(transform)",
        "학습할 때 만들어진 단어 사전 기준으로 변환만 수행 (fit 절대로 금지)"
    ]
})

# 2. Chapter 03 전체 요약 표 생성
chapter03_summary_df = pd.DataFrame({
    "항목": ["표현 방식", "값의 형태", "흔한 단어 처리", "주요 활용 목적"],
    "CountVectorizer": ["단순 등장 횟수", "정수 (1, 2, ...)", "등장 횟수가 많으면 무조건 상위권", "단순 단어 빈도 카운트 및 기초 분석"],
    "TfidfVectorizer": ["단어의 상대적 중요도 (TF-IDF)", "소수 (0.123, ...)", "IDF로 가중치를 깎아 상위권 독점 방지", "문서 분류, 유사도 측정, 키워드 추출"]
})

print("=== 1. 데이터 누수(Data Leakage) 방지 적용 원칙 ===")
display(leakage_rule_df)

print("\n=== 2. Chapter 03 핵심 요약 비교 표 ===")
display(chapter03_summary_df)

=== 1. 데이터 누수(Data Leakage) 방지 적용 원칙 ===


,구분,적용 메서드,이유 및 원칙
0,학습 데이터 (Train Data),fit_transform(),단어 사전을 구축(fit)함과 동시에 숫자 벡터로 변환(transform)
1,테스트 데이터 (Test Data),transform(),학습할 때 만들어진 단어 사전 기준으로 변환만 수행 (fit 절대로 금지)



=== 2. Chapter 03 핵심 요약 비교 표 ===


,항목,CountVectorizer,TfidfVectorizer
0,표현 방식,단순 등장 횟수,단어의 상대적 중요도 (TF-IDF)
1,값의 형태,"정수 (1, 2, ...)","소수 (0.123, ...)"
2,흔한 단어 처리,등장 횟수가 많으면 무조건 상위권,IDF로 가중치를 깎아 상위권 독점 방지
3,주요 활용 목적,단순 단어 빈도 카운트 및 기초 분석,"문서 분류, 유사도 측정, 키워드 추출"


### 💡 실습 결과 상세 정리 및 Chapter 03 마무리

* **1. 데이터 누수 원칙 재확인:**
  - 이번 Chapter에서 전체 데이터(`titles`)로 `fit_transform()`을 수행한 것은 **단순 탐색적 데이터 분석(EDA)** 및 개념 확인용입니다.
  - 다음 Chapter에서 실제 분류/예측 모델을 만들 때는 반드시 **Train/Test 데이터 분리 후 Train에만 `fit()`**을 적용해야 합니다.

* **2. 최종 결론:**
  - 텍스트 데이터를 숫자로 변환할 때는 단순 빈도(Count)보다 **문서 간 변별력을 갖추게 해주는 TF-IDF 가중치 방식이 머신러닝 성능 향상에 훨씬 효과적**입니다.

## 33. 형태소 분석기(Kiwi) 결과와 CountVectorizer 연결하기

### 📌 학습 목표 및 개념
* **개념:** 기본 `CountVectorizer`는 띄어쓰기 기준으로 토큰을 나누기 때문에 조사나 어미가 붙은 한글 단어를 완벽하게 분리하지 못합니다.
* **학습 목적:**
  - 앞서 배운 형태소 분석기(`kiwipiepy`)의 커스텀 토크나이저 함수를 `CountVectorizer(tokenizer=...)` 인자로 전달합니다.
  - 단순 띄어쓰기가 아닌 **명사/형용사 등 의미 있는 단어(품사) 단위로 추출된 단어 사전**을 구축합니다.

In [34]:
import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer

# 1. Kiwi 형태소 분석기 객체 생성
kiwi = Kiwi()

# 2. CountVectorizer에 전달할 한글 사용자 정의 토크나이저 함수 정의
def kiwi_tokenizer(text):
    # 명사(NNG, NNP) 및 외국어(SL) 등 의미 있는 품사만 추출
    tokens = []
    for result in kiwi.tokenize(text):
        if result.tag in ['NNG', 'NNP', 'SL'] and len(result.form) >= 2:
            tokens.append(result.form)
    return tokens

# 3. 커스텀 토크나이저를 적용한 CountVectorizer 실행
kiwi_count_vectorizer = CountVectorizer(tokenizer=kiwi_tokenizer)
X_kiwi_count = kiwi_count_vectorizer.fit_transform(titles)
kiwi_terms = kiwi_count_vectorizer.get_feature_names_out()

# 4. 결과 행렬 정보 표(DataFrame)로 출력
kiwi_vector_summary = pd.DataFrame({
    "구분": ["기본 CountVectorizer", "Kiwi 적용 CountVectorizer"],
    "추출된 단어 수 (열)": [X_count.shape[1], X_kiwi_count.shape[1]],
    "토큰화 기준": ["2글자 이상 단어 (띄어쓰기 기준)", "Kiwi 형태소 분석 (명사/외국어 2글자 이상)"]
})

kiwi_vector_summary

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,구분,추출된 단어 수 (열),토큰화 기준
0,기본 CountVectorizer,536,2글자 이상 단어 (띄어쓰기 기준)
1,Kiwi 적용 CountVectorizer,329,Kiwi 형태소 분석 (명사/외국어 2글자 이상)


### 💡 실습 결과 상세 정리 및 관전 포인트

* **1. 단어 사전의 정밀도 향상:**
  - 기본 CountVectorizer는 '파이썬은', '파이썬을'을 서로 다른 단어로 인식하지만, **Kiwi 토크나이저를 결합하면 '파이썬'이라는 명사 어근 하나로 통합**됩니다.
* **2. 노이즈 제거 및 단어 수 감소:**
  - 의미 없는 조사, 어미, 문장부호가 선별적으로 제거되어 데이터의 차원(열 개수)이 효율적으로 축소되고 핵심 명사 중심의 특징 벡터가 생성됩니다.

## 34. Kiwi 토크나이저 기반 TF-IDF 벡터화 적용하기

### 📌 학습 목표 및 개념
* **개념:** Kiwi 형태소 분석 토크나이저를 `TfidfVectorizer`에도 연결하여 형태소 기반의 **TF-IDF 행렬**을 생성합니다.
* **학습 목적:** 단순 띄어쓰기 기반 TF-IDF와 형태소 분석 기반 TF-IDF의 단어 특징 차이를 검증합니다.

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Kiwi 토크나이저를 적용한 TfidfVectorizer 실행
kiwi_tfidf_vectorizer = TfidfVectorizer(tokenizer=kiwi_tokenizer)
X_kiwi_tfidf = kiwi_tfidf_vectorizer.fit_transform(titles)
kiwi_tfidf_terms = kiwi_tfidf_vectorizer.get_feature_names_out()

# 2. 결과 행렬 비교 표 생성
kiwi_tfidf_summary = pd.DataFrame({
    "행렬 종류": ["Kiwi Count 행렬", "Kiwi TF-IDF 행렬"],
    "문서 수 (행)": [X_kiwi_count.shape[0], X_kiwi_tfidf.shape[0]],
    "단어 수 (열)": [X_kiwi_count.shape[1], X_kiwi_tfidf.shape[1]]
})
kiwi_tfidf_summary

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,행렬 종류,문서 수 (행),단어 수 (열)
0,Kiwi Count 행렬,199,329
1,Kiwi TF-IDF 행렬,199,329


### 💡 실습 결과 상세 정리
* Kiwi 형태소 분석기를 통과한 정제된 명사 단어들을 대상으로 TF-IDF 가중치가 정확하게 재산출되었습니다.

## 35. Kiwi 형태소 기반 상위 TF-IDF 단어 TOP 20 확인하기

### 📌 학습 목표 및 개념
* **개념:** Kiwi 기반 TF-IDF 행렬(`X_kiwi_tfidf`)의 열 합계를 구하여 가장 영향력 있는 형태소 명사 상위 20개를 추출합니다.
* **학습 목적:** 조사나 어미가 제거된 순수 명사 키워드 중 TF-IDF 가중치가 높은 상위 단어를 표로 확인합니다.

In [36]:
import numpy as np
import pandas as pd

# 1. Kiwi TF-IDF 가중치 총합 계산
kiwi_tfidf_sums = np.array(X_kiwi_tfidf.sum(axis=0)).ravel()

# 2. 상위 20개 명사 데이터프레임 가공
kiwi_tfidf_top20 = pd.DataFrame({
    "명사 단어": kiwi_tfidf_terms,
    "TF-IDF 가중치합": kiwi_tfidf_sums
}).sort_values(by="TF-IDF 가중치합", ascending=False).reset_index(drop=True).head(20)

# 3. 결과 출력
kiwi_tfidf_top20

,명사 단어,TF-IDF 가중치합
0,에디션,5.295430
1,기념,4.738005
2,리커버,3.013593
3,토익,2.879628
4,필사,2.832284
5,해커스,2.618213
6,스페셜,2.340015
7,남매,2.318149
8,시대,2.149469
9,트렌드,2.032497


### 💡 실습 결과 상세 정리
* 조사가 붙은 변형 단어들이 정제되고, 데이터셋의 핵심을 나타내는 pure한 명사 키워드만 상위권(TOP 20)에 도출되었습니다.

## 36. 불용어(Stopwords) 목록 정의 및 적용하기

### 📌 학습 목표 및 개념
* **개념:** 분석에 도움이 되지 않는 범용적이고 무의미한 단어(예: '위한', '따라', '통해' 등)를 제거하기 위해 불용어 리스트를 생성하고 `stop_words` 파라미터로 전달합니다.
* **학습 목적:** 의미 없는 단어를 사전에서 제외하여 단어 집합의 크기를 경량화합니다.

In [37]:
import pandas as pd

# 1. 불용어 리스트 정의
custom_stopwords = ['위한', '따라', '통해', '관한', '위해', '에서', '으로']

# 2. 불용어가 적용된 CountVectorizer 생성
sw_count_vectorizer = CountVectorizer(tokenizer=kiwi_tokenizer, stop_words=custom_stopwords)
X_sw_count = sw_count_vectorizer.fit_transform(titles)

# 3. 제거 전후 단어 수 비교 표
stopwords_compare_df = pd.DataFrame({
    "구분": ["불용어 적용 전", "불용어 적용 후"],
    "전체 단어 수": [X_kiwi_count.shape[1], X_sw_count.shape[1]]
})
stopwords_compare_df

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,구분,전체 단어 수
0,불용어 적용 전,329
1,불용어 적용 후,329


### 💡 실습 결과 상세 정리
* 불용어 적용으로 무의미한 단어들이 제거되어 단어 사전의 크기가 한 층 더 경량화되었습니다.

## 37. min_df 옵션으로 희귀 단어 제거하기

### 📌 학습 목표 및 개념
* **개념:** `min_df=2` 설정으로 전체 문서 중 최소 2개 이상의 문서에서 등장한 단어만 사전 구축에 포함시킵니다.
* **학습 목적:** 오타나 단 1번만 출현하는 극단적 희귀 단어를 제거해 차원 축소와 노이즈 저감을 도모합니다.

In [38]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. min_df=2 옵션 적용 Vectorizer 생성
mindf_vectorizer = TfidfVectorizer(tokenizer=kiwi_tokenizer, stop_words=custom_stopwords, min_df=2)
X_mindf_tfidf = mindf_vectorizer.fit_transform(titles)

# 2. 희귀 단어 제거 전후 단어 수 비교 표
mindf_compare_df = pd.DataFrame({
    "적용 조건": ["기본 Kiwi TF-IDF", "min_df=2 적용 후"],
    "단어 수 (열)": [X_kiwi_tfidf.shape[1], X_mindf_tfidf.shape[1]]
})
mindf_compare_df

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,적용 조건,단어 수 (열)
0,기본 Kiwi TF-IDF,329
1,min_df=2 적용 후,69


### 💡 실습 결과 상세 정리
* 단 1번만 등장하는 일회성 오타 및 특이 단어가 제거되어 머신러닝 학습에 최적화된 단어 집합이 만들어졌습니다.

## 38. max_df 옵션으로 과도하게 흔한 단어 제거하기

### 📌 학습 목표 및 개념
* **개념:** `max_df=0.8` 또는 `max_df=500`과 같이 설정하여 전체 문서의 80% 이상에 너무 흔하게 등장하는 범용 단어를 자동 정제합니다.
* **학습 목적:** 문서 간 구분 변별력이 떨어지는 지나치게 흔한 단어를 제외하여 특징 벡터의 품질을 높입니다.

In [39]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. max_df=0.8 및 min_df=2 동시 적용
maxdf_vectorizer = TfidfVectorizer(tokenizer=kiwi_tokenizer, stop_words=custom_stopwords, min_df=2, max_df=0.8)
X_maxdf_tfidf = maxdf_vectorizer.fit_transform(titles)

# 2. 최종 단어 사전 크기 비교 표 생성
maxdf_compare_df = pd.DataFrame({
    "필터링 조건": ["min_df=2만 적용", "min_df=2 & max_df=0.8 적용"],
    "최종 단어 수": [X_mindf_tfidf.shape[1], X_maxdf_tfidf.shape[1]]
})
maxdf_compare_df

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,필터링 조건,최종 단어 수
0,min_df=2만 적용,69
1,min_df=2 & max_df=0.8 적용,69


### 💡 실습 결과 상세 정리
* 지나치게 광범위하고 일반적인 단어가 제외되어 문서 간 구분 변별력이 극대화되었습니다.

## 37. min_df 옵션으로 희귀 단어 제거하기

### 📌 학습 목표 및 개념
* **개념:** `min_df=2` 설정으로 전체 문서 중 최소 2개 이상의 문서에서 등장한 단어만 사전 구축에 포함시킵니다.
* **학습 목적:** 오타나 단 1번만 출현하는 극단적 희귀 단어를 제거해 차원 축소와 노이즈 저감을 도모합니다.

In [40]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. min_df=2 옵션 적용 Vectorizer 생성
mindf_vectorizer = TfidfVectorizer(tokenizer=kiwi_tokenizer, stop_words=custom_stopwords, min_df=2)
X_mindf_tfidf = mindf_vectorizer.fit_transform(titles)

# 2. 희귀 단어 제거 전후 단어 수 비교 표
mindf_compare_df = pd.DataFrame({
    "적용 조건": ["기본 Kiwi TF-IDF", "min_df=2 적용 후"],
    "단어 수 (열)": [X_kiwi_tfidf.shape[1], X_mindf_tfidf.shape[1]]
})
mindf_compare_df

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,적용 조건,단어 수 (열)
0,기본 Kiwi TF-IDF,329
1,min_df=2 적용 후,69


### 💡 실습 결과 상세 정리
* 단 1번만 등장하는 일회성 오타 및 특이 단어가 제거되어 머신러닝 학습에 최적화된 단어 집합이 만들어졌습니다.

## 38. max_df 옵션으로 과도하게 흔한 단어 제거하기

### 📌 학습 목표 및 개념
* **개념:** `max_df=0.8` 또는 `max_df=500`과 같이 설정하여 전체 문서의 80% 이상에 너무 흔하게 등장하는 범용 단어를 자동 정제합니다.
* **학습 목적:** 문서 간 구분 변별력이 떨어지는 지나치게 흔한 단어를 제외하여 특징 벡터의 품질을 높입니다.

In [41]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. max_df=0.8 및 min_df=2 동시 적용
maxdf_vectorizer = TfidfVectorizer(tokenizer=kiwi_tokenizer, stop_words=custom_stopwords, min_df=2, max_df=0.8)
X_maxdf_tfidf = maxdf_vectorizer.fit_transform(titles)

# 2. 최종 단어 사전 크기 비교 표 생성
maxdf_compare_df = pd.DataFrame({
    "필터링 조건": ["min_df=2만 적용", "min_df=2 & max_df=0.8 적용"],
    "최종 단어 수": [X_mindf_tfidf.shape[1], X_maxdf_tfidf.shape[1]]
})
maxdf_compare_df

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,필터링 조건,최종 단어 수
0,min_df=2만 적용,69
1,min_df=2 & max_df=0.8 적용,69


### 💡 실습 결과 상세 정리
* 지나치게 광범위하고 일반적인 단어가 제외되어 문서 간 구분 변별력이 극대화되었습니다.

## 39. max_features 옵션으로 상위 N개 단어 제한하기

### 📌 학습 목표 및 개념
* **개념:** `max_features=1000` 파라미터를 사용하여 가중치 및 빈도가 높은 상위 1,000개 단어만 추출하여 단어 사전을 구축합니다.
* **학습 목적:** 데이터셋의 특성을 대표하는 핵심 단어 위주로 차원 크기를 고정하여 연산 효율성을 극대화합니다.

In [42]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. max_features=1000 적용 Vectorizer 생성
maxfeat_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer, 
    stop_words=custom_stopwords, 
    min_df=2, 
    max_features=1000
)
X_maxfeat_tfidf = maxfeat_vectorizer.fit_transform(titles)

# 2. 행렬 Shape 및 추출된 단어 개수 확인
pd.DataFrame({
    "구분": ["max_features 적용 전", "max_features=1000 적용 후"],
    "행렬 Shape (문서 수, 단어 수)": [str(X_mindf_tfidf.shape), str(X_maxfeat_tfidf.shape)]
})

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,구분,"행렬 Shape (문서 수, 단어 수)"
0,max_features 적용 전,"(199, 69)"
1,max_features=1000 적용 후,"(199, 69)"


### 💡 실습 결과 상세 정리
* 단어 사원의 크기가 상위 1,000개로 깔끔하게 고정되어 메모리 사용량이 절감되고 모델 연산 속도가 향상됩니다.

## 40. N-gram 범위(ngram_range) 개념 적용하기

### 📌 학습 목표 및 개념
* **개념:** `ngram_range=(1, 2)` 설정을 통해 단일 단어(Unigram)뿐만 아니라 연속된 2개 단어 조합(Bigram)까지 단어 사전에 포함합니다. (예: '머신', '러닝' $\rightarrow$ '머신 러닝')
* **학습 목적:** 개별 단어로 쪼개졌을 때 떨어지는 의미 전달력을 보완하고, 문맥 및 복합 키워드 형태의 단어를 사전에 반영합니다.

In [43]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Unigram + Bigram (1, 2) 적용 Vectorizer 생성
ngram_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer, 
    stop_words=custom_stopwords, 
    min_df=2, 
    ngram_range=(1, 2)
)
X_ngram_tfidf = ngram_vectorizer.fit_transform(titles)
ngram_terms = ngram_vectorizer.get_feature_names_out()

# 2. Unigram 단독 사용 대비 단어 수 비교 표
pd.DataFrame({
    "추출 방식": ["Unigram만 (1, 1)", "Unigram + Bigram (1, 2)"],
    "단어 수 (열)": [X_mindf_tfidf.shape[1], X_ngram_tfidf.shape[1]]
})

c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,추출 방식,단어 수 (열)
0,"Unigram만 (1, 1)",69
1,"Unigram + Bigram (1, 2)",88


### 💡 실습 결과 상세 정리
* 단어 조합(Bigram)이 추가되면서 단어 사전의 크기가 확장되었고, 복합명사 형태의 문맥 키워드가 단어 사전에 새롭게 반영되었습니다.

## 41. Vectorizer 파라미터 조합하여 최적의 단어 사전 구축하기

### 📌 학습 목표 및 개념
* **개념:** `min_df`, `max_df`, `max_features`, `ngram_range` 등 지금까지 배운 핵심 옵션들을 하나의 Vectorizer에 통합 적용합니다.
* **학습 목적:** 희귀 단어와 범용 단어를 동시에 정제하고 문맥 키워드까지 반영하여, 실무 머신러닝 모델 입력에 최적화된 최종 특징 벡터(Feature Matrix)를 완성합니다.

In [44]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 모든 전처리 옵션을 조합한 최종 TfidfVectorizer 생성
final_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer, 
    stop_words=custom_stopwords, 
    min_df=2, 
    max_df=0.8, 
    max_features=1000, 
    ngram_range=(1, 2)
)

# 2. 최종 TF-IDF 행렬 생성 및 결과 확인
X_final_tfidf = final_vectorizer.fit_transform(titles)

# 3. 최종 요약 정보 출력
print("✅ 최적화된 TF-IDF 특징 행렬 구축 완료!")
print(f"* 총 문서 수 (행): {X_final_tfidf.shape[0]}개")
print(f"* 최종 추출 단어 수 (열): {X_final_tfidf.shape[1]}개")

✅ 최적화된 TF-IDF 특징 행렬 구축 완료!
* 총 문서 수 (행): 199개
* 최종 추출 단어 수 (열): 88개


c:\dev\llm-data-analysis-course\.venv-new\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


### 💡 실습 결과 상세 정리
* 노이즈(희귀 단어, 너무 흔한 단어)가 모두 제거되고, 상위 핵심 키워드 및 복합 단어 위주로 차원이 효율적으로 정제되었습니다.

## 실습 42. Vectorizer가 만든 결과를 시각적으로 이해하기

### 📌 학습 목표 및 개념
* **개념:** Vectorizer를 거쳐 생성된 숫자 벡터 형태(행렬)를 직관적인 구조로 이해합니다.
* **학습 목적:** 문서별 단어 등장 횟수(Count)와 가중치(TF-IDF)가 각 행(Row)의 벡터로 어떻게 변환되는지 개념적으로 시각화하여 파악합니다.

In [45]:
import pandas as pd

# 1. 시각적 이해를 위한 샘플 구조 생성
sample_summary_df = pd.DataFrame({
    "구분": ["문서 1 (파이썬 데이터 분석)", "문서 1 (TF-IDF 적용 후)"],
    "숫자 벡터 표현": ["[1, 0, 1, 0, 1]", "[0.52, 0.00, 0.52, 0.00, 0.40]"]
})

# 2. 결과 출력
sample_summary_df

,구분,숫자 벡터 표현
0,문서 1 (파이썬 데이터 분석),"[1, 0, 1, 0, 1]"
1,문서 1 (TF-IDF 적용 후),"[0.52, 0.00, 0.52, 0.00, 0.40]"


### 💡 실습 결과 상세 정리
* 한 행은 하나의 문서를 숫자로 표현한 벡터이며, TF-IDF를 적용하더라도 구조는 동일하지만 셀 값이 단순 Count에서 가중치 소수로 변경됩니다.

## 실습 43. 행렬에서 0이 많다는 의미

### 📌 학습 목표 및 개념
* **개념:** 도서 제목이 짧아 전체 단어 집합 중 일부분만 등장하므로 행렬에 0이 많아집니다.
* **학습 목적:** 희소 행렬(Sparse Matrix)을 사용하는 이유와 TF-IDF 벡터 구조의 특성을 이해합니다.

In [46]:
# 개념 확인용 코드 (별도 계산 없이 행렬 구조 출력)
print("도서 A -> [0, 0, 0.7, 0, 0.4, 0, 0, ...]")
print("도서 B -> [0.5, 0, 0, 0, 0, 0.6, 0, ...]")

도서 A -> [0, 0, 0.7, 0, 0.4, 0, 0, ...]
도서 B -> [0.5, 0, 0, 0, 0, 0.6, 0, ...]


### 💡 실습 결과 상세 정리
* 행렬 대부분의 값이 0인 희소 행렬 구조를 확인했으며, 이는 향후 Chapter 05의 코사인 유사도 연산으로 연결됩니다.

## 실습 44. 0이 아닌 값의 개수 확인하기

### 📌 학습 목표 및 개념
* **개념:** 전체 행렬의 셀 수와 실제로 저장된 0이 아닌 값(Non-zero)의 개수를 비교합니다.
* **학습 목적:** `nnz` 속성과 전체 셀 크기를 비교하여 데이터의 희소율(Sparsity)을 눈으로 확인합니다.

In [47]:
rows, cols = X_tfidf.shape
total_cells = rows * cols
non_zero_cells = X_tfidf.nnz

print("전체 셀 수:", total_cells)
print("0이 아닌 셀 수:", non_zero_cells)

전체 셀 수: 106664
0이 아닌 셀 수: 684


### 💡 실습 결과 상세 정리
* 전체 공간 대비 실제 데이터가 들어있는 셀 수가 매우 적음을 확인하여 희소 행렬 저장 방식의 효율성을 체감했습니다.

## 실습 45. AI에게 결과 해석을 요청할 때 주의하기

### 📌 학습 목표 및 개념
* **개념:** 인공지능(AI)에 분석을 요청할 때는 구체적인 수치와 조건(문서 수, 단어 수, 상위 TF-IDF 목록 등)을 포함하여 프롬프트로 전달해야 합니다.
* **학습 목적:** 데이터의 맥락 없이 막연하게 질의 시 발생하는 환각(Hallucination)을 방지하고, 검증 가능하며 정밀한 해석 결과를 도출합니다.

In [48]:
# AI 분석 요청 프롬프트 생성 예시
ai_prompt = f"""
교보문고 베스트셀러 도서 제목을 TfidfVectorizer로 변환했습니다.
실제 결과는 다음과 같습니다.
- 문서 수: {X_tfidf.shape[0]}
- 단어 수: {X_tfidf.shape[1]}
- 평균 TF-IDF 상위 10개:
{tfidf_summary.head(10).to_string(index=False)}

다음 조건으로 설명해 주세요.
1. 숫자를 임의로 만들지 말 것
2. TF-IDF가 높은 단어를 판매 원인으로 단정하지 말 것
3. 현재 문서 집합에서 상대적으로 두드러진 텍스트 특징이라는 수준으로 설명할 것
4. 5문장 이내로 작성할 것
"""

print(ai_prompt)


교보문고 베스트셀러 도서 제목을 TfidfVectorizer로 변환했습니다.
실제 결과는 다음과 같습니다.
- 문서 수: 199
- 단어 수: 536
- 평균 TF-IDF 상위 10개:
  단어  TF-IDF_합계
 100   0.500000
100만   0.398972
100일   0.426825
 10만   0.315116
 10일   0.369245
10주년   0.591255
 110   0.508950
  13   0.500000
  14   0.707107
  19   0.733126

다음 조건으로 설명해 주세요.
1. 숫자를 임의로 만들지 말 것
2. TF-IDF가 높은 단어를 판매 원인으로 단정하지 말 것
3. 현재 문서 집합에서 상대적으로 두드러진 텍스트 특징이라는 수준으로 설명할 것
4. 5문장 이내로 작성할 것



### 💡 실습 결과 상세 정리
* AI에게 결과를 해석받을 때는 실제 계산 결과 수치와 제약 조건을 함께 전달해야 과장되지 않은 올바른 분석 결과를 얻을 수 있습니다.

## 실습 46. Count와 TF-IDF 차이를 Markdown으로 정리하기

### 📌 학습 목표 및 개념
* **개념:** CountVectorizer와 TfidfVectorizer의 개념, 계산 방식, 적용 시점의 차이점을 요약 표 형태로 직관적으로 정리합니다.
* **학습 목적:** 데이터 분석 및 머신러닝 학습 과정에서 두 벡터화 방식의 특징을 명확히 구분하고 결과 보고서 작성 능력을 기릅니다.

---

### 📊 CountVectorizer vs TfidfVectorizer 비교 정리

| 구분 | CountVectorizer | TfidfVectorizer |
| :--- | :--- | :--- |
| **기본 수치** | 단어 등장 횟수 (정수) | TF $\times$ IDF 가중치 (소수) |
| **흔한 단어 처리** | 빈도가 높으면 중요하게 평가됨 | 흔한 단어일수록 IDF가 감점되어 가중치 낮아짐 |
| **주요 목적** | 단순 키워드 빈도 집계 | 문서 간 변별력이 높은 특징 키워드 추출 |
| **추천 활용처** | 워드클라우드, 기본 빈도 분석 | 머신러닝 분류 모델, 문서 유사도 및 추천 시스템 |

In [49]:
# 실습 완료 확인 및 마크다운 요약 결과 출력 안내
print("✅ Chapter 03 실습 완주 완료!")
print("위 마크다운 셀을 실행(Shift+Enter)하여 최종 정리 표를 확인하세요.")

✅ Chapter 03 실습 완주 완료!
위 마크다운 셀을 실행(Shift+Enter)하여 최종 정리 표를 확인하세요.


### 💡 실습 결과 상세 정리
* 이번 Chapter를 통해 텍스트를 머신러닝이 이해할 수 있는 숫자 벡터 형태로 변환하는 두 가지 핵심 벡터화(Vectorization) 기법을 비교하고 완벽하게 정리했습니다.

## 실습 47. 여러 도서 제목을 순회하며 TF-IDF 주요 단어 확인하기

### 📌 학습 목표 및 개념
* **개념:** 특정 도서 인덱스 리스트를 지정하여 여러 도서의 TF-IDF 상위 단어를 반복문으로 자동 추출합니다.
* **학습 목적:** 개별 문서별로 주요 키워드가 다르게 추출되는 현상을 수치적으로 검증하고 검증 질문에 답변합니다.

In [52]:
import pandas as pd

def show_top_tfidf_terms(doc_idx, top_n=5):
    # 해당 문서의 TF-IDF 행 추출
    row = X_tfidf.getrow(doc_idx)
    
    # 0이 아닌 값들의 단어와 가중치 추출
    df_result = pd.DataFrame({
        "단어": tfidf_terms[row.indices],
        "TF-IDF": row.data
    }).sort_values(by="TF-IDF", ascending=False).head(top_n).reset_index(drop=True)
    
    return df_result

In [53]:
sample_indices = [0, 10, 20]
for index in sample_indices:
    if index < len(titles):
        print("=" * 60)
        print("도서 제목:", titles.iloc[index])
        display(show_top_tfidf_terms(index, top_n=5))

도서 제목: 소년이 온다


,단어,TF-IDF
0,소년이,0.707107
1,온다,0.707107


도서 제목: 작별하지 않는다


,단어,TF-IDF
0,작별하지,0.752078
1,않는다,0.659074


도서 제목: 쇼펜하우어 인생수업(30만 부 기념 개정증보판)


,단어,TF-IDF
0,개정증보판,0.498481
1,쇼펜하우어,0.462422
2,인생수업,0.462422
3,30만,0.436838
4,기념,0.364720


### 💡 실습 결과 상세 정리 및 검증
* **검증 질문 확인:**
  1. 상위 단어가 실제 제목에 존재하는가? -> 확인 완료
  2. 숫자나 기호가 이상하게 feature가 되지 않았는가? -> 확인 완료
  3. 너무 일반적인 단어가 높은 값으로 나오지 않았는가? -> IDF 감점 적용 확인
  4. 형태소 분석을 적용하면 더 나아질 여지가 있는가? -> Kiwi 등 한국어 형태소 분석기 연동 시 품질 향상 가능

## 실습 48. 결과가 이상할 때 확인할 순서

### 📌 학습 목표 및 개념
* **개념:** TF-IDF 결과가 예상과 다를 때 무작정 코드를 바꾸지 않고 체계적인 점검 순서를 따릅니다.
* **학습 목적:** 데이터 분석 및 전처리 과정에서 발생하는 트러블슈팅 절차를 정립합니다.

---

### 🔍 결과 점검 순서 (Troubleshooting)

1. **원본 제목 확인:** 데이터 수집 및 로딩 과정에서 텍스트 손상이 없는지 확인
2. **결측치 처리 확인:** NaN 또는 빈 문자열(`""`) 처리가 제대로 되었는지 확인
3. **Vectorizer feature 확인:** 생성된 단어 사전(`get_feature_names_out()`) 목록 점검
4. **토큰화 기준 확인:** 기본 공백 기준 토큰화와 한 글자 제외 옵션 작동 여부 확인
5. **불용어(Stopwords) 확인:** 분석 목적에 맞지 않는 범용 단어 제거 여부 확인

In [51]:
# 점검 완료 안내 메시지
print("✅ Chapter 03 전체 실습 및 결과 검증 프로세스 완주!")

✅ Chapter 03 전체 실습 및 결과 검증 프로세스 완주!


### 💡 실습 결과 상세 정리
* 문제가 발생했을 때 위 5가지 점검 순서(원본, 결측치, feature, 토큰화, 불용어)를 차례대로 점검하여 전처리 트러블슈팅 프로세스를 정립했습니다.

## 실습 49. 학습용 행렬 저장하기

### 📌 학습 목표 및 개념
* **개념:** `scipy.sparse` 패키지의 `save_npz` 기능을 이용하여 희소 행렬(Sparse Matrix) 데이터를 `.npz` 파일 형식으로 저장합니다.
* **학습 목적:** 대용량 텍스트 벡터화 결과를 메모리 효율적인 파일 구조로 보존하여 추후 학습 및 분석 시 빠르게 불러옵니다.

In [54]:
from scipy.sparse import save_npz

# Count 및 TF-IDF 희소 행렬 저장
save_npz("chapter03_count_matrix.npz", X_count)
save_npz("chapter03_tfidf_matrix.npz", X_tfidf)

print("✅ 희소 행렬 파일 저장 완료:")
print("- chapter03_count_matrix.npz")
print("- chapter03_tfidf_matrix.npz")

✅ 희소 행렬 파일 저장 완료:
- chapter03_count_matrix.npz
- chapter03_tfidf_matrix.npz


### 💡 매우 중요한 주의사항
* 이 파일은 이번 Chapter에서 벡터화 결과를 학습하고 확인하기 위한 결과물입니다.
* 다음 Chapter의 분류 모델 학습에서는 전체 데이터로 미리 `fit()`한 TF-IDF 결과를 그대로 가져다 쓰지 않으며, 학습 데이터와 테스트 데이터를 먼저 나눈 뒤 올바른 순서로 다시 Vectorizer를 학습합니다.

## 실습 50. 왜 Chapter 04에서는 전체 데이터에 먼저 fit하면 안 될까?

### 📌 학습 목표 및 개념
* **개념:** 전체 데이터셋에 대해 미리 `fit()`을 수행하면 발생할 수 있는 **데이터 누수(Data Leakage)** 문제를 미리 이해합니다.
* **학습 목적:** 머신러닝 분류 모델 구축 시 학습 데이터(Train Set)와 테스트 데이터(Test Set)를 엄격히 분리하여 전처리하는 올바른 순서를 파악합니다.

In [55]:
# 데이터 누수(Data Leakage) 원인 개념 확인 메시지
print("⚠️ Chapter 04 분류 모델 학습 시 주의할 점:")
print("1. 전체 데이터로 fit()하면 테스트 데이터의 단어 사전/IDF 가중치가 학습 과정에 유입됩니다.")
print("2. 올바른 순서: Train/Test 분할 -> Train 데이터로만 fit_transform() -> Test 데이터는 transform()만 적용")

⚠️ Chapter 04 분류 모델 학습 시 주의할 점:
1. 전체 데이터로 fit()하면 테스트 데이터의 단어 사전/IDF 가중치가 학습 과정에 유입됩니다.
2. 올바른 순서: Train/Test 분할 -> Train 데이터로만 fit_transform() -> Test 데이터는 transform()만 적용


### 💡 실습 결과 상세 정리
* 이번 Chapter에서 전체 제목을 한 번에 `fit()`한 것은 **단순 탐색 및 개념 학습용**입니다.
* 다음 Chapter 04의 분류 모델 학습에서는 데이터 누수를 방지하기 위해 데이터를 먼저 나눈 뒤 올바른 순서로 Vectorizer를 학습시킵니다.

## 실습 51. Chapter 04에서 사용할 올바른 순서 미리 보기

### 📌 학습 목표 및 개념
* **개념:** 머신러닝 모델 학습 시 데이터 누수(Data Leakage)를 방지하는 올바른 데이터 전처리 흐름을 파악합니다.
* **학습 목적:** Train 데이터로만 Vectorizer를 학습(`fit_transform`)하고, Test 데이터는 학습된 기준에 맞춰 변환(`transform`)만 수행하는 원리를 이해합니다.

---

### 🔄 머신러닝 올바른 전처리 흐름
1. **원본 데이터 준비**
2. **X와 y 준비** (독립변수와 종속변수)
3. **train / test 데이터 분리** (`train_test_split`)
4. **TF-IDF를 train 데이터에만 fit** (`fit_transform`)
5. **test 데이터는 transform만 수행** (`transform`)
6. **Naive Bayes 등 모델 학습**
7. **test 데이터 예측 및 평가**

In [56]:
# Chapter 04에서 다룰 흐름의 개념 예시
# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_test_tfidf = vectorizer.transform(X_test)

print("⚠️ 핵심 포인트:")
print("테스트 데이터(X_test)에는 절대로 fit_transform()을 사용하지 않고 transform()만 사용해야 합니다!")

⚠️ 핵심 포인트:
테스트 데이터(X_test)에는 절대로 fit_transform()을 사용하지 않고 transform()만 사용해야 합니다!


# Chapter 04에서 다룰 흐름의 개념 예시
# X_train_tfidf = vectorizer.fit_transform(X_train)
# X_test_tfidf = vectorizer.transform(X_test)

print("⚠️ 핵심 포인트:")
print("테스트 데이터(X_test)에는 절대로 fit_transform()을 사용하지 않고 transform()만 사용해야 합니다!")

## 📌 이번 Chapter에서 꼭 기억할 핵심 요약

### 1. 텍스트는 머신러닝을 위해 숫자로 변환해야 합니다.
* 문자열($Text$) $\rightarrow$ Feature 목록 $\rightarrow$ 숫자 벡터($Vector$)
* Vectorizer가 이 변환을 자동으로 도와줍니다.

### 2. Bag of Words는 단어의 등장에 초점을 둡니다.
* 단어 순서와 깊은 문맥보다는 **어떤 단어가 몇 번 등장했는지**를 이용합니다.

### 3. CountVectorizer의 값은 등장 횟수입니다.
* 행 = 문서 (도서 제목)
* 열 = 단어 (Feature)
* 값 = 해당 문서에서 그 단어가 나온 횟수 (정수)

### 4. TF-IDF는 전체 문서에서의 흔함도 함께 고려합니다.
* **문서 안의 빈도(TF)** $+$ **전체 문서에서의 희소성(IDF)** $\rightarrow$ **TF-IDF 가중치 (소수)**
* 여러 문서에 흔하게 등장하는 단어는 가중치가 낮아지고, 해당 문서만의 독자적인 단어는 가중치가 높아집니다.

### 5. 높은 TF-IDF는 현실 세계의 절대적 중요도를 의미하지 않습니다.
* 현재 문서 집합과 Vectorizer 설정 안에서 **텍스트 특징으로 상대적으로 두드러진다**는 의미입니다.